In [1]:
import numpy as np
import numpy.testing as npt

from scipy import stats
from scipy.stats import t,ttest_ind
from scipy.stats import f
from scipy.stats import f_oneway
from scipy.stats import pearsonr

import statsmodels.api as sm
from statsmodels.regression._prediction import get_prediction
from statsmodels.stats.outliers_influence import OLSInfluence,MLEInfluence
from statsmodels.graphics.gofplots import qqplot_2samples,ProbPlot,qqplot
import pandas as pd
from patsy import dmatrices
from numpy.testing import assert_almost_equal, assert_allclose
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns

import os

# some_file.py
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, r'C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\statemodelsStudy')
from olsRegressionAnalysis import dispAnalysisOfVariance, tableDispFormatt,getInvOfProductMat,\
                                  getRegressionEqn,\
                                  dispReghressionAnalysis,norm_scalling,getCorrelation,\
                                  get_variance_inflation_factors

In [2]:
path = os.path.join(os.getcwd(), 'DataSet', 'TABLE_3_2_DeliveryTimeData.csv')
df = pd.read_csv(path)
df.columns = ['Obs','DlvrTImeY','NumCaseX1','DstX2']

print(df.columns)

Index(['Obs', 'DlvrTImeY', 'NumCaseX1', 'DstX2'], dtype='object')


In [3]:
y, X = dmatrices(
                 'DlvrTImeY ~ NumCaseX1 + DstX2', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()

tableDispFormatt('Estimator/Cofficient')
print(res.params)
tableDispFormatt('Estimator/Cofficient Std err')
print(res.bse)
tableDispFormatt('Estimator/Cofficient CI')

=============================== Estimator/Cofficient =======================================
Intercept    2.341231
NumCaseX1    1.615907
DstX2        0.014385
dtype: float64
=============================== Estimator/Cofficient Std err ===============================
Intercept    1.096730
NumCaseX1    0.170735
DstX2        0.003613
dtype: float64
=============================== Estimator/Cofficient CI ====================================


CI of β s

    confidance cofficient = 1 – α

    βj ± tα/2,n-p SE(βj)

# Get CI method 1:

### Confidance interval:

conf_int(alpha=0.05, cols=None)

alpha:

    The significance level for the confidence interval. 

    The default alpha = .05 returns a 95% confidence interval

cols:

     pecifies which confidence intervals to return.
     
     Example res.conf_int(cols=(1,2))


In [4]:
print(res.conf_int(alpha = 0.05)) #95% CI

                  0         1
Intercept  0.066752  4.615710
NumCaseX1  1.261825  1.969990
DstX2      0.006892  0.021878


# Get CI method 2:

In [5]:

se = res.bse
tSig = t.isf(q = (0.05/2), df = res.df_resid, loc=0, scale=1)
BETAj = res.params.values
ci_u = BETAj + (tSig*se)
ci_l = BETAj - (tSig*se)
ci = np.column_stack((ci_l, ci_u))
print(ci)

[[0.06675199 4.6157103 ]
 [1.26182466 1.96998976]
 [0.00689174 0.02187791]]



Get T values using Bonferroni Correction.

https://www.youtube.com/watch?v=HLzS5wPqWR0

3.4.3 Simultaneous Confidence Intervals on

      Regression Coefficients
      
CI of β with Bonferroni Corrections

    confidance cofficient = 1 – α/p
    
    βj ± tα/2p,n-p SE(βj)


In [6]:
tableDispFormatt('CI by using Bonferroni Correction')
se = res.bse


=============================== CI by using Bonferroni Correction ==========================


In [7]:
# for 95% CI
alfa = 0.05/(2*res.df_model)
tSig = t.isf(q = alfa, df = res.df_resid, loc=0, scale=1)
BETAj = res.params.values
ci_u = BETAj + (tSig*se)
ci_l = BETAj - (tSig*se)
ci = np.column_stack((ci_l, ci_u))
print(ci)

[[-0.29692338  4.97938567]
 [ 1.20520902  2.0266054 ]
 [ 0.00569365  0.02307601]]
